# Ispezione del dataset SFT

Questo notebook carica un campione di esempi SFT da `data/processed/sft.sample.jsonl`, mostra i conteggi per dominio e stampa alcuni esempi.

Se il campione non esiste viene generato **offline** con il `MockTeacher` (nessuna rete, nessuna GPU), cosi' il notebook resta sempre eseguibile.

In [ ]:
import os
import sys
from collections import Counter

def find_repo_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(d, 'src', 'italian_llm')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return os.path.abspath(os.getcwd())
        d = parent

REPO_ROOT = find_repo_root()
SRC = os.path.join(REPO_ROOT, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

print('Repo root:', REPO_ROOT)

In [ ]:
from italian_llm.utils.io import read_jsonl
from italian_llm.data.prompts import SYNTH_PROMPTS
from italian_llm.data.synthetic import MockTeacher, synthesize_batch

SAMPLE_PATH = os.path.join(REPO_ROOT, 'data', 'processed', 'sft.sample.jsonl')

USER_PROMPTS = {
    'qa': 'Spiega in breve cos e la fotosintesi clorofilliana.',
    'summary': 'Riassumi in 3 frasi la storia della Repubblica di Venezia.',
    'rewrite': 'Riscrivi in modo professionale: ciao volevo sapere se va bene cosi.',
    'coding': 'Scrivi una funzione Python che calcola i numeri primi fino a n.',
    'email': 'Scrivi una email per chiedere un preventivo a un fornitore.',
    'helpdesk': 'Il mio ordine non e ancora arrivato, cosa devo fare?',
    'faq': 'Posso modificare l indirizzo di spedizione dopo un ordine?',
    'admin': 'Quali documenti servono per rinnovare la carta di identita?',
    'dialog': 'Vorrei pianificare un viaggio di tre giorni a Roma, mi aiuti?',
    'doc': 'Prepara la struttura di una relazione tecnica su un progetto software.',
}

def build_sample(path, n_per_domain=2):
    tasks = []
    for domain in SYNTH_PROMPTS:
        base_user = USER_PROMPTS.get(domain, 'Aiutami con una richiesta di esempio.')
        for k in range(n_per_domain):
            difficulty = 'hard' if k % 2 else 'medium'
            tasks.append({
                'id': f'{domain}-{k:02d}',
                'domain': domain,
                'difficulty': difficulty,
                'user': base_user,
            })
    n = synthesize_batch(tasks, MockTeacher(), path)
    print(f'Generati {n} esempi offline in {path}')

if not os.path.exists(SAMPLE_PATH):
    print('Campione assente: lo genero offline col MockTeacher...')
    build_sample(SAMPLE_PATH)

rows = list(read_jsonl(SAMPLE_PATH))
print(f'Esempi caricati: {len(rows)}')

In [ ]:
import statistics

counts = Counter(r.get('domain', 'sconosciuto') for r in rows)
print('Conteggio esempi per dominio:')
for domain, n in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])):
    print(f'  {domain:10s} {n:3d}')

qs = [float(r.get('quality_score', 0.0)) for r in rows]
it = [float(r.get('italian_score', 0.0)) for r in rows]
if rows:
    print()
    print(f'quality_score medio: {statistics.mean(qs):.3f}')
    print(f'italian_score medio: {statistics.mean(it):.3f}')

safety = Counter(r.get('safety_tag', 'allow') for r in rows)
print('Distribuzione safety_tag:', dict(safety))

In [ ]:
from italian_llm.data.schema import validate_record

def show_example(rec):
    print('id:', rec.get('id'), '| dominio:', rec.get('domain'), '| difficolta:', rec.get('difficulty'))
    for m in rec.get('messages', []):
        content = m.get('content', '')
        snippet = content if len(content) <= 200 else content[:200] + '...'
        print('  [' + str(m.get('role')) + '] ' + snippet)
    ok, reason = validate_record(rec)
    print('  valido:', 'si' if ok else 'no (' + reason + ')')
    print('-' * 72)

for rec in rows[:3]:
    show_example(rec)